# DocSem PP-OCRv5 cloud CPU shard

이 노트북은 Kaggle 또는 Colab의 CPU 런타임에서 validation PDF의 한 shard만 OCR합니다. 입력 묶음에는 PDF와 label-free manifest만 포함하며 정답, evidence, 후보 답안, 기존 OCR 전사는 넣지 않습니다. `SHARD_INDEX`만 다르게 설정한 독립 실행 결과를 Mac Studio에서 검증·병합합니다.

In [ ]:
import os
from pathlib import Path

SHARD_COUNT = 8
SHARD_INDEX = 0  # 각 노트북 실행마다 0..7 중 하나만 지정
MAX_RETRY_ROUNDS = 2
REPO_URL = "https://github.com/choco9966/docinsights-2026.git"
REPO_REF = "feature/8"
EXPECTED_REPO_SHA = "b5958def41c1d600c5f190d76be9386f615b7f71"
BUNDLE_NAME = "docsem-validation-ocr-input.tar.gz"
EXPECTED_BUNDLE_SHA256 = "9fb35e81feead385fedd3a5bd66ca780ca2aaee5b92b2247f75114cfae642967"
EXPECTED_MANIFEST_SHA256 = "08bb8ef1948bdbb69ceddfc669d31adf7002707cdd149937b04615dae0eb2d3b"

if not 0 <= SHARD_INDEX < SHARD_COUNT:
    raise ValueError(f"SHARD_INDEX must be in 0..{SHARD_COUNT - 1}")
IS_KAGGLE = Path("/kaggle/working").is_dir()
WORK_ROOT = Path("/kaggle/working" if IS_KAGGLE else "/content") / "docsem-ocr-cloud"
REPO_ROOT = WORK_ROOT / "repo"
EXTRACT_ROOT = WORK_ROOT / "input"
OUTPUT_ROOT = WORK_ROOT / "output"
PERSIST_ROOT = (
    Path(os.environ["DOCSEM_PERSIST_DIR"]) if os.environ.get("DOCSEM_PERSIST_DIR") else OUTPUT_ROOT
)
explicit_bundle = os.environ.get("DOCSEM_BUNDLE_PATH")
search_roots = [Path("/kaggle/input"), Path("/content"), Path.cwd()]
bundle_candidates = (
    [Path(explicit_bundle)]
    if explicit_bundle
    else [path for root in search_roots if root.exists() for path in root.rglob(BUNDLE_NAME)]
)
if len(bundle_candidates) != 1:
    raise RuntimeError(
        f"Set DOCSEM_BUNDLE_PATH or expose exactly one {BUNDLE_NAME}; found {bundle_candidates}"
    )
BUNDLE_PATH = bundle_candidates[0].resolve()
for directory in (WORK_ROOT, EXTRACT_ROOT, OUTPUT_ROOT, PERSIST_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
print(
    {
        "platform": "kaggle" if IS_KAGGLE else "colab",
        "bundle": str(BUNDLE_PATH),
        "shard": f"{SHARD_INDEX}/{SHARD_COUNT}",
    }
)

## 환경 설치

Kaggle에서는 Internet 옵션을 켜고, Colab에서는 표준 CPU 런타임을 사용합니다. 패키지와 모델 revision은 아래 값으로 고정하며 실제 환경은 마지막 산출물에 다시 기록합니다.

In [ ]:
import subprocess
import sys

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "poppler-utils", "git"], check=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "paddlepaddle==3.2.0",
        "-i",
        "https://www.paddlepaddle.org.cn/packages/stable/cpu/",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "paddleocr==3.3.2",
        "paddlex==3.3.13",
        "huggingface-hub==0.34.4",
    ],
    check=True,
)
if not (REPO_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", EXPECTED_REPO_SHA], check=True)
git_sha = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], check=True, capture_output=True, text=True
).stdout.strip()
if git_sha != EXPECTED_REPO_SHA:
    raise RuntimeError(f"Repository SHA mismatch: {git_sha}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(REPO_ROOT)],
    check=True,
)

In [ ]:
import hashlib
import tarfile


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def safe_extract(archive_path: Path, destination: Path) -> None:
    destination = destination.resolve()
    with tarfile.open(archive_path, "r:gz") as archive:
        for member in archive.getmembers():
            target = (destination / member.name).resolve()
            if not target.is_relative_to(destination) or member.issym() or member.islnk():
                raise RuntimeError(f"Unsafe archive member: {member.name}")
        archive.extractall(destination)


safe_extract(BUNDLE_PATH, EXTRACT_ROOT)
BUNDLE_ROOT = EXTRACT_ROOT / "bundle"
MANIFEST = BUNDLE_ROOT / "manifest.jsonl"
DOCUMENTS_ROOT = BUNDLE_ROOT / "documents-root"
bundle_sha256 = sha256_file(BUNDLE_PATH)
manifest_sha256 = sha256_file(MANIFEST)
if bundle_sha256 != EXPECTED_BUNDLE_SHA256 or manifest_sha256 != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError({"bundle_sha256": bundle_sha256, "manifest_sha256": manifest_sha256})
print({"bundle_sha256": bundle_sha256, "manifest_sha256": manifest_sha256})

In [ ]:
from huggingface_hub import snapshot_download

DET_REPO = "PaddlePaddle/PP-OCRv5_mobile_det"
DET_REVISION = "0d63e78e2b680928f6b1747d76a08db6e645efb7"
REC_REPO = "PaddlePaddle/en_PP-OCRv5_mobile_rec"
REC_REVISION = "267c36e24c331595590fe7bd72bde2436fd286f2"
MODEL_ROOT = WORK_ROOT / "models"
DET_DIR = Path(
    snapshot_download(DET_REPO, revision=DET_REVISION, local_dir=MODEL_ROOT / "detector")
)
REC_DIR = Path(
    snapshot_download(REC_REPO, revision=REC_REVISION, local_dir=MODEL_ROOT / "recognizer")
)
print({"detector": str(DET_DIR), "recognizer": str(REC_DIR)})

## 결정적 shard 실행과 체크포인트

한 실행은 한 shard와 한 결과 파일만 소유하고 PP-OCR pipeline을 shard 전체에서 유지합니다. 매 문서 뒤 원자적 checkpoint를 `PERSIST_ROOT`에 기록하므로 같은 셀을 다시 실행하면 이미 성공한 문서는 건너뜁니다. 첫 단계는 미처리 문서를 먼저 완료하고, 그 뒤 실패 문서만 최대 2회 재시도하며 진전이 없으면 종료합니다. 다른 VM에서는 기존 checkpoint를 이어 쓰지 않습니다. Colab에서 Drive를 쓸 때는 실행 전에 Drive를 mount하고 `DOCSEM_PERSIST_DIR`을 Drive 내부의 새 실험 디렉터리로 지정하세요.

In [ ]:
import importlib.metadata
import json
import platform
from datetime import UTC, datetime

SHARD_DIR = WORK_ROOT / f"shards-{SHARD_COUNT:02}"
subprocess.run(
    [
        sys.executable,
        "-m",
        "docinsights_ocr",
        "cloud-shard",
        str(MANIFEST),
        str(SHARD_DIR),
        "--shard-count",
        str(SHARD_COUNT),
    ],
    check=True,
)
SHARD_MANIFEST = SHARD_DIR / f"manifest-shard-{SHARD_INDEX:02}-of-{SHARD_COUNT:02}.jsonl"
RESULT = PERSIST_ROOT / f"result-shard-{SHARD_INDEX:02}-of-{SHARD_COUNT:02}.jsonl"
CHECKPOINT = RESULT.with_name(RESULT.name + ".retry-checkpoint")
SESSION_STATE = PERSIST_ROOT / f"session-shard-{SHARD_INDEX:02}-of-{SHARD_COUNT:02}.json"
STARTED_AT = datetime.now(UTC).isoformat()
boot_id_path = Path("/proc/sys/kernel/random/boot_id")
boot_id = boot_id_path.read_text().strip() if boot_id_path.is_file() else platform.node()
session_fingerprint = hashlib.sha256(
    json.dumps(
        {"boot_id": boot_id, "platform": platform.platform(), "python": platform.python_version()},
        sort_keys=True,
    ).encode()
).hexdigest()
packages = {
    name: importlib.metadata.version(name)
    for name in ["paddlepaddle", "paddleocr", "paddlex", "huggingface-hub"]
}
session = {
    "session_fingerprint": session_fingerprint,
    "repository_sha": git_sha,
    "bundle_sha256": bundle_sha256,
    "manifest_sha256": manifest_sha256,
    "packages": packages,
}
if (RESULT.is_file() or CHECKPOINT.is_file()) and not SESSION_STATE.is_file():
    raise RuntimeError(
        "Checkpoint exists without session identity; start a new experiment directory"
    )
if SESSION_STATE.is_file() and json.loads(SESSION_STATE.read_text()) != session:
    raise RuntimeError(
        "Checkpoint belongs to another VM or software cohort; start a new experiment directory"
    )
session_tmp = SESSION_STATE.with_name(SESSION_STATE.name + ".tmp")
session_tmp.write_text(json.dumps(session, ensure_ascii=False, indent=2, sort_keys=True) + "\n")
session_tmp.replace(SESSION_STATE)
expected_count = sum(1 for line in SHARD_MANIFEST.read_text().splitlines() if line.strip())


def completed_count(path: Path) -> int:
    return sum(1 for line in path.read_text().splitlines() if line.strip()) if path.is_file() else 0


def load_records(path: Path) -> list[dict]:
    return (
        [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
        if path.is_file()
        else []
    )


def active_result_path() -> Path:
    return CHECKPOINT if CHECKPOINT.is_file() else RESULT


base_command = [
    sys.executable,
    "-m",
    "docinsights_ocr",
    "run",
    str(SHARD_MANIFEST),
    str(RESULT),
    "--engine",
    "paddleocr",
    "--dpi",
    "200",
    "--documents-root",
    str(DOCUMENTS_ROOT),
    "--paddle-detection-model-dir",
    str(DET_DIR),
    "--paddle-recognition-model-dir",
    str(REC_DIR),
    "--paddle-detection-model-revision",
    DET_REVISION,
    "--paddle-recognition-model-revision",
    REC_REVISION,
    "--pipeline-revision",
    git_sha,
    "--timeout-seconds",
    "300",
    "--retry-failed",
    "--resume",
]

while completed_count(RESULT) < expected_count:
    before = completed_count(active_result_path())
    subprocess.run(base_command, cwd=REPO_ROOT, check=True)
    after = completed_count(RESULT)
    if after <= before:
        raise RuntimeError(f"No checkpoint progress: {before}/{expected_count}")
    print({"completed": after, "expected": expected_count})
for retry_round in range(1, MAX_RETRY_ROUNDS + 1):
    failed_before = {
        record["instance_id"] for record in load_records(RESULT) if record["status"] == "failed"
    }
    if not failed_before:
        break
    subprocess.run(base_command, cwd=REPO_ROOT, check=True)
    failed_after = {
        record["instance_id"] for record in load_records(RESULT) if record["status"] == "failed"
    }
    print(
        {
            "retry_round": retry_round,
            "failed_before": len(failed_before),
            "failed_after": len(failed_after),
        }
    )
    if failed_after == failed_before:
        break
records = load_records(RESULT)
record_ids = [record["instance_id"] for record in records]
if len(records) != expected_count or len(record_ids) != len(set(record_ids)):
    raise RuntimeError(
        {"records": len(records), "unique_ids": len(set(record_ids)), "expected": expected_count}
    )
FINAL_FAILED = [record["instance_id"] for record in records if record["status"] == "failed"]
print({"records": len(records), "failed": FINAL_FAILED})

In [ ]:
RUNTIME = PERSIST_ROOT / f"runtime-shard-{SHARD_INDEX:02}-of-{SHARD_COUNT:02}.json"
FREEZE = PERSIST_ROOT / f"pip-freeze-shard-{SHARD_INDEX:02}-of-{SHARD_COUNT:02}.txt"
git_status = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "status", "--porcelain"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
runtime = {
    "schema_version": "1.0",
    "platform_role": "kaggle-cpu" if IS_KAGGLE else "colab-cpu",
    "platform": platform.platform(),
    "machine": platform.machine(),
    "python": platform.python_version(),
    "repository_sha": git_sha,
    "repository_dirty": bool(git_status),
    "pipeline_revision": git_sha,
    "bundle_sha256": bundle_sha256,
    "manifest_sha256": manifest_sha256,
    "shard_manifest_sha256": sha256_file(SHARD_MANIFEST),
    "shard_count": SHARD_COUNT,
    "shard_index": SHARD_INDEX,
    "result_sha256": sha256_file(RESULT),
    "timeout_seconds": 300.0,
    "session_fingerprint": session_fingerprint,
    "started_at": STARTED_AT,
    "finished_at": datetime.now(UTC).isoformat(),
    "runtime_version": {
        key: os.environ.get(key) for key in ["KAGGLE_KERNEL_RUN_TYPE", "COLAB_RELEASE_TAG"]
    },
    "detector": {"repo": DET_REPO, "revision": DET_REVISION},
    "recognizer": {"repo": REC_REPO, "revision": REC_REVISION},
    "packages": packages,
    "record_count": len(records),
    "failed_count": len(FINAL_FAILED),
}
RUNTIME.write_text(json.dumps(runtime, ensure_ascii=False, indent=2, sort_keys=True) + "\n")
FREEZE.write_text(
    subprocess.run(
        [sys.executable, "-m", "pip", "freeze"], check=True, capture_output=True, text=True
    ).stdout
)
print(runtime)
if FINAL_FAILED:
    raise RuntimeError(f"Shard has failed OCR records after bounded retries: {FINAL_FAILED}")

## 회수할 파일

`result-shard-XX-of-08.jsonl`, `runtime-shard-XX-of-08.json`, `pip-freeze-shard-XX-of-08.txt` 세 파일을 Mac Studio의 같은 실험 디렉터리로 모읍니다. Kaggle과 Colab 결과는 실행 환경 fingerprint가 다르면 한 결과로 섞지 않고 별도 cohort로 비교합니다.